# SDG 11: Sustainable cities and communities

The SDG 11 (Sustainable cities and communities) index of the SDG Atlas of Bolivia 2026 and its 7 indicators, overcrowding, toilets, adequate housing, public transport, solid-waste spending and collection, and PM2.5 pollution, for every municipality near 2012 and near 2024.

This notebook gathers every code box of the indicator's page, [https://quarcs-lab.github.io/gdo4bolivia/indicators/municipalities343-sdg11/](https://quarcs-lab.github.io/gdo4bolivia/indicators/municipalities343-sdg11/) (GeoDevelopment Observatory of Bolivia), so you can work with the data in one place. Run **Setup** first; then any section runs on its own (Runtime › Run all runs everything).

## Ask Gemini, Colab's AI

Colab has an AI assistant, Gemini. Open it with the Gemini button (✦) in Colab's toolbar, or click **Generate** in an empty code cell, and paste one of these questions. It writes and explains code that builds on the data loaded below. Run Setup first, and check its answers against the variables' descriptions on the page.

- "Map overcrowded households (sdg11_1_thm) in 2024 and list the 10 municipalities with the highest share."
- "How much did adequate housing (sdg11_1_ava) improve between ~2012 and ~2024? Show the change by department."
- "Plot solid-waste collection (sdg11_6_srr) against public spending on solid waste (sdg11_6_ipr) in 2024, with a log scale for spending."
- "Where is PM2.5 concentration (sdg11_6_ica) highest? Map it for 2024 and check whether it clusters spatially with geo.moran."
- "Which municipalities have public transport (sdg11_2_atc) at all? Compare the Sustainable cities index (index_sdg11) of those with and without it."


## Variables

The columns of `sdg11`, one row per unit and year. Each one's full description (how it was built, its source and reference years) is under Variables on the page, and in `geo.dictionary("sdg11")`.

| Variable | Label | Unit |
| --- | --- | --- |
| `geo_id` | Municipality ID |  |
| `year` | Period |  |
| `mun_id` | INE municipality code |  |
| `index_sdg11` | Sustainable cities and communities index (0-100) | index (0-100) |
| `sdg11_1_thm` | Overcrowded households (%) | % of households |
| `sdg11_1_ssb` | Households without a toilet (%) | % of households |
| `sdg11_1_ava` | Adequate housing (% of dwellings) | % of dwellings |
| `sdg11_2_atc` | Public-transport seats (per 1,000 people) | seats per 1,000 people |
| `sdg11_6_ipr` | Public spending on solid waste (Bs per person per year) | bolivianos per person per year |
| `sdg11_6_srr` | Solid-waste collection (% of dwellings) | % of dwellings |
| `sdg11_6_ica` | PM2.5 concentration (µg/m³) | µg/m³ |


## Setup

Installs the packages every section uses and imports them once.

In [ ]:
!pip install -q geo4bolivia geolibre
import geo4bolivia as geo
import plotly.express as px
import pandas as pd
import geopandas as gpd
from geolibre import Map

## Load the data with the package

In [ ]:
# The SDG 11 index (0-100, higher is better), ~2024
gdf = geo.load("sdg11", variables="index_sdg11", years=2024, geometry=True)
px.choropleth_map(gdf, geojson=gdf.geometry, locations=gdf.index,
          color="index_sdg11", hover_name="name",
          center={"lat": -16.3, "lon": -64}, zoom=4,
          labels={"index_sdg11": "Index (0-100)"},
          title="SDG 11: Sustainable cities and communities index, ~2024").show()

# The two periods: initial (~2012) and final (~2024) values
df = geo.load("sdg11", variables="index_sdg11")
wide = df.pivot(index="geo_id", columns="year", values="index_sdg11")
wide = wide.add_prefix("y")
wide["name"] = geo.units("map343").set_index("geo_id")["name"]
px.scatter(wide, x="y2012", y="y2024", hover_name="name",
           labels={"y2012": "Index, ~2012", "y2024": "Index, ~2024"},
           title="SDG 11 index, ~2012 and ~2024")

## Load the data without the package (plain URLs)

In [ ]:
# No geo4bolivia needed: every file is a plain URL
BASE = "https://raw.githubusercontent.com/quarcs-lab/gdo4bolivia/main/"
df = pd.read_csv(BASE + "data/sdg11/map343/sdg11.csv")  # geo_id, year, mun_id, ...
municipalities = gpd.read_file(BASE + "geo/boundaries/map343.geojson")

index = df.loc[df["year"] == 2024, ["geo_id", "index_sdg11"]]
gdf = municipalities.merge(index, on="geo_id")
px.choropleth_map(gdf, geojson=gdf.geometry, locations=gdf.index,
          color="index_sdg11", hover_name="name",
          center={"lat": -16.3, "lon": -64}, zoom=4,
          labels={"index_sdg11": "Index (0-100)"},
          title="SDG 11: Sustainable cities and communities index, ~2024")

## Load the data from the wide GeoPackage (one row per municipality)

In [ ]:
# No geo4bolivia needed: one row per municipality, a column per variable
# and year (<variable>_<year>), with boundaries
BASE = "https://raw.githubusercontent.com/quarcs-lab/gdo4bolivia/main/"
gdf = gpd.read_file(BASE + "data/sdg11/map343/sdg11_wide.gpkg",
                    layer="sdg11")   # or sdg11_wide.geojson (lighter)
px.choropleth_map(gdf, geojson=gdf.geometry, locations=gdf.index,
          color="index_sdg11_2024", hover_name="name",
          center={"lat": -16.3, "lon": -64}, zoom=4,
          labels={"index_sdg11_2024": "Sustainable cities and communities index (0-100)"},
          title="Sustainable cities and communities index (0-100), ~2024")

## Load the data in GeoLibre (the wide GeoJSON)

In [ ]:
# GeoLibre in the notebook: the wide GeoJSON, one feature per municipality
# and a column per variable and year (<variable>_<year>); restyle any of
# them in the Style panel
BASE = "https://raw.githubusercontent.com/quarcs-lab/gdo4bolivia/main/"
m = Map(center=[-64.5, -16.5], zoom=4.5, height="600px")
m.add_choropleth(BASE + "data/sdg11/map343/sdg11_wide.geojson",
                 column="index_sdg11_2024", name="Sustainable cities and communities index (0-100), ~2024",
                 strokeColor="#ffffff", strokeWidth=0.3)
m

## Map the change, ~2012 to ~2024

In [ ]:
df = geo.load("sdg11", variables="index_sdg11")
wide = df.pivot(index="geo_id", columns="year", values="index_sdg11")
wide["change"] = wide[2024] - wide[2012]
gdf = geo.boundaries("map343").merge(wide["change"].reset_index(),
                                           on="geo_id")
px.choropleth_map(gdf, geojson=gdf.geometry, locations=gdf.index,
          color="change", hover_name="name",
          center={"lat": -16.3, "lon": -64}, zoom=4,
          labels={"change": "Change in the index (points)"},
          title="SDG 11: change in the index, ~2012 to ~2024")

## Recompute the index from its indicators

In [ ]:
df = geo.load("sdg11", years=2024).set_index("geo_id")
bounds = {"sdg11_1_thm": (68.2, 12.6), "sdg11_1_ssb": (100, 0), "sdg11_1_ava": (0,
          100), "sdg11_2_atc": (0, 342), "sdg11_6_ipr": (0, 287.7),
          "sdg11_6_srr": (0, 100), "sdg11_6_ica": (31, 5)}  # (worst, best)
scores = pd.DataFrame({v: 100 * (df[v] - w) / (b - w)
                       for v, (w, b) in bounds.items()}).clip(0, 100)
df["computed"] = scores.mean(axis=1)  # skips missing indicators
df["name"] = geo.units("map343").set_index("geo_id")["name"]
px.scatter(df, x="computed", y="index_sdg11", hover_name="name",
           labels={"computed": "Computed from the indicators",
                   "index_sdg11": "Index (0-100)"},
           title="Recomputed vs published SDG 11 index, ~2024")

## Which indicators moved

In [ ]:
df = geo.load("sdg11")
bounds = {"sdg11_1_thm": (68.2, 12.6), "sdg11_1_ssb": (100, 0), "sdg11_1_ava": (0,
          100), "sdg11_2_atc": (0, 342), "sdg11_6_ipr": (0, 287.7),
          "sdg11_6_srr": (0, 100), "sdg11_6_ica": (31, 5)}  # (worst, best)
labels = geo.dictionary("sdg11").set_index("variable")["label"]
rows = [{"indicator": labels[v], "period": f"~{year}",
         "score": (100 * (part[v] - w) / (b - w)).clip(0, 100).mean()}
        for year, part in df.groupby("year") for v, (w, b) in bounds.items()]
px.scatter(pd.DataFrame(rows), x="score", y="indicator", color="period",
           labels={"score": "Mean rescaled score (0-100)", "indicator": ""},
           title="SDG 11 indicators, ~2012 and ~2024")

## Rank the departments

In [ ]:
df = geo.load("sdg11", variables="index_sdg11")
dep = geo.aggregate(df, to="department",
                    rules={"index_sdg11": "weighted_mean:population"},
                    weights=geo.load("sdg", variables="population"))
dep["department"] = dep["geo_id"].map(
    geo.units("department").set_index("geo_id")["name"])
dep["period"] = "~" + dep["year"].astype(str)
px.scatter(dep.sort_values("index_sdg11"), x="index_sdg11", y="department",
           color="period",
           labels={"index_sdg11": "Index (population-weighted)"},
           title="SDG 11 index by department, ~2012 and ~2024")

## License and how to cite

The data are published under [Creative Commons Attribution 4.0 International License](https://creativecommons.org/licenses/by/4.0/); their sources keep their own licenses:

- Lykke E. Andersen and Fabiana Argandoña (SDSN Bolivia, Universidad Privada Boliviana): [Creative Commons Attribution 4.0 International License (licensed by SDSN Bolivia)](https://creativecommons.org/licenses/by/4.0/)
- SDSN Bolivia: [Creative Commons Attribution 4.0 International License (licensed by SDSN Bolivia)](https://creativecommons.org/licenses/by/4.0/)

If you use these data, please cite, in this order, the original source, the observatory, which processes and maintains them, and the other sources:

1. Andersen, L. E., & Argandoña, F. (2026). Atlas de los ODS en Bolivia 2026: Brechas y tendencias del desarrollo sostenible en las Entidades Territoriales Autónomas, 2012–2024. La Paz: Universidad Privada Boliviana, SDSN Bolivia.
2. Mendez, C. (2026). GeoDevelopment Observatory of Bolivia. https://quarcs-lab.github.io/gdo4bolivia/
3. SDSN Bolivia (2025). Límites Municipales Bolivia 2025 [Archivo shapefile]. https://sdsnbolivia.org/datos-espaciales/
